In [ ]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Aprendizaje básico](intro.html) \|\|
[Inicio rápido](quickstart_tutorial.html) \|\|
[Tensores](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\|
[Transformaciones](transforms_tutorial.html) \|\| [Construcción del
Modelo](buildmodel_tutorial.html) \|\| **Autograd** \|\|
[Optimización](optimization_tutorial.html) \|\| [Guardar y Cargar
Modelo](saveloadrun_tutorial.html)

Diferenciación automática con `torch.autograd`
==============================================

Al entrenar redes neuronales, el algoritmo más utilizado es la
**retropropagación** (*back propagation*). En este algoritmo, los
parámetros (pesos del modelo) se ajustan según el **gradiente** de la
función de pérdida con respecto al parámetro dado.

Para calcular esos gradientes, PyTorch cuenta con un motor de
diferenciación incorporado llamado `torch.autograd`. Este soporta el
cálculo automático del gradiente para cualquier grafo computacional.

Considera la red neuronal más simple de una capa, con entrada `x`,
parámetros `w` y `b`, y alguna función de pérdida. Se puede definir en
PyTorch de la siguiente manera:


In [ ]:
import torch

x = torch.ones(5)  # input tensor
y = torch.zeros(3)  # expected output
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w)+b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

Tensores, Funciones y Grafo Computacional
=========================================

Este código define el siguiente **grafo computacional**:

![](https://pytorch.org/tutorials/_static/img/basics/comp-graph.png)

En esta red, `w` y `b` son **parámetros** que necesitamos optimizar. Por
tanto, necesitamos poder calcular los gradientes de la función de pérdida
con respecto a esas variables. Para ello, establecemos la propiedad
`requires_grad` de esos tensores.


<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTA:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>Puedes establecer el valor de <code>requires_grad</code> al crear un tensor, o más tarde usando el método <code>x.requires_grad_(True)</code>.</p>

</div>



Una función que aplicamos a los tensores para construir el grafo
computacional es en realidad un objeto de la clase `Function`. Este
objeto sabe cómo calcular la función en la dirección *forward*, y también
cómo calcular su derivada durante el paso de *retropropagación*. Una
referencia a la función de retropropagación se almacena en la propiedad
`grad_fn` de un tensor. Puedes encontrar más información sobre `Function`
[en la documentación](https://pytorch.org/docs/stable/autograd.html#function).


In [ ]:
print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")

Cálculo de Gradientes
=====================

Para optimizar los pesos de los parámetros en la red neuronal, necesitamos
calcular las derivadas de nuestra función de pérdida con respecto a los
parámetros, es decir, necesitamos $\frac{\partial loss}{\partial w}$ y
$\frac{\partial loss}{\partial b}$ bajo valores fijos de `x` e `y`. Para
calcular esas derivadas, llamamos a `loss.backward()` y luego recuperamos
los valores de `w.grad` y `b.grad`:


In [ ]:
loss.backward()
print(w.grad)
print(b.grad)

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTA:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<ul>
<li>Solo podemos obtener las propiedades <code>grad</code> para los nodos hoja del grafo computacional, que tienen la propiedad <code>requires_grad</code> establecida en <code>True</code>. Para todos los demás nodos del grafo, los gradientes no estarán disponibles.</li>
<li>Solo podemos realizar cálculos de gradiente usando <code>backward</code> una vez sobre un grafo dado, por razones de rendimiento. Si necesitamos hacer varias llamadas a <code>backward</code> sobre el mismo grafo, debemos pasar <code>retain_graph=True</code> a la llamada de <code>backward</code>.</li>
</ul>

</div>



Desactivar el Seguimiento de Gradientes
========================================

Por defecto, todos los tensores con `requires_grad=True` rastrean su
historial computacional y soportan el cálculo de gradientes. Sin embargo,
hay casos en los que no necesitamos hacer eso; por ejemplo, cuando hemos
entrenado el modelo y solo queremos aplicarlo a datos de entrada, es decir,
solo queremos hacer computaciones *forward* a través de la red. Podemos
detener el seguimiento de computaciones envolviendo nuestro código con un
bloque `torch.no_grad()`:


In [ ]:
z = torch.matmul(x, w)+b
print(z.requires_grad)

with torch.no_grad():
    z = torch.matmul(x, w)+b
print(z.requires_grad)

Otra forma de lograr el mismo resultado es usar el método `detach()` sobre
el tensor:


In [ ]:
z = torch.matmul(x, w)+b
z_det = z.detach()
print(z_det.requires_grad)

Hay razones por las que podrías querer desactivar el seguimiento de gradientes:

-   Para marcar algunos parámetros de tu red neuronal como **parámetros
    congelados**.
-   Para **acelerar los cálculos** cuando solo realizas el paso forward,
    ya que las operaciones sobre tensores que no rastrean gradientes son
    más eficientes.


Más sobre Grafos Computacionales
=================================

Conceptualmente, autograd mantiene un registro de los datos (tensores) y
de todas las operaciones ejecutadas (junto con los nuevos tensores
resultantes) en un grafo acíclico dirigido (DAG, por sus siglas en inglés)
compuesto por objetos
[Function](https://pytorch.org/docs/stable/autograd.html#torch.autograd.Function).
En este DAG, las hojas son los tensores de entrada y las raíces son los
tensores de salida. Al recorrer este grafo desde las raíces hasta las
hojas, los gradientes se pueden calcular automáticamente usando la regla
de la cadena.

En un paso forward, autograd hace dos cosas simultáneamente:

-   ejecuta la operación solicitada para calcular el tensor resultante,
-   mantiene la *función de gradiente* de la operación en el DAG.

El paso backward se inicia cuando se llama a `.backward()` sobre la raíz
del DAG. `autograd` entonces:

-   calcula los gradientes a partir de cada `.grad_fn`,
-   los acumula en el atributo `.grad` del tensor correspondiente,
-   usando la regla de la cadena, los propaga hasta los tensores hoja.

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTA:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>Un aspecto importante a tener en cuenta es que el grafo se recrea desde cero; después de cada llamada a <code>.backward()</code>, autograd comienza a poblar un nuevo grafo. Esto es precisamente lo que te permite usar sentencias de control de flujo en tu modelo; puedes cambiar la forma, el tamaño y las operaciones en cada iteración si es necesario.</p>

</div>



Lectura opcional: Gradientes de Tensores y Productos Jacobianos
===============================================================

En muchos casos tenemos una función de pérdida escalar y necesitamos
calcular el gradiente con respecto a algunos parámetros. Sin embargo, hay
casos en que la función de salida es un tensor arbitrario. En ese caso,
PyTorch permite calcular el llamado **producto jacobiano**, y no el
gradiente real.

Para una función vectorial $\vec{y}=f(\vec{x})$, donde
$\vec{x}=\langle x_1,\dots,x_n\rangle$ y
$\vec{y}=\langle y_1,\dots,y_m\rangle$, el gradiente de $\vec{y}$ con
respecto a $\vec{x}$ viene dado por la **matriz Jacobiana**:

$$\begin{aligned}
J=\left(\begin{array}{ccc}
   \frac{\partial y_{1}}{\partial x_{1}} & \cdots & \frac{\partial y_{1}}{\partial x_{n}}\\
   \vdots & \ddots & \vdots\\
   \frac{\partial y_{m}}{\partial x_{1}} & \cdots & \frac{\partial y_{m}}{\partial x_{n}}
   \end{array}\right)
\end{aligned}$$

En lugar de calcular la matriz Jacobiana en sí, PyTorch permite calcular
el **Producto Jacobiano** $v^T\cdot J$ para un vector de entrada dado
$v=(v_1 \dots v_m)$. Esto se logra llamando a `backward` con $v$ como
argumento. El tamaño de $v$ debe ser igual al tamaño del tensor original
con respecto al cual queremos calcular el producto:


In [ ]:
inp = torch.eye(4, 5, requires_grad=True)
out = (inp+1).pow(2).t()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"First call\n{inp.grad}")
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nSecond call\n{inp.grad}")
inp.grad.zero_()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nCall after zeroing gradients\n{inp.grad}")

Observa que cuando llamamos a `backward` por segunda vez con el mismo
argumento, el valor del gradiente es diferente. Esto ocurre porque al
realizar la retropropagación, PyTorch **acumula los gradientes**, es
decir, el valor de los gradientes calculados se suma a la propiedad `grad`
de todos los nodos hoja del grafo computacional. Si quieres calcular los
gradientes correctamente, debes poner a cero la propiedad `grad` antes.
En el entrenamiento real, un *optimizador* nos ayuda a hacer esto.


<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTA:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>Anteriormente llamábamos a la función <code>backward()</code> sin parámetros. Esto es esencialmente equivalente a llamar a <code>backward(torch.tensor(1.0))</code>, que es una forma útil de calcular los gradientes en el caso de una función con valor escalar, como la pérdida durante el entrenamiento de redes neuronales.</p>

</div>



------------------------------------------------------------------------


Lecturas adicionales
====================

-   [Mecánica de Autograd](https://pytorch.org/docs/stable/notes/autograd.html)
